# Classify your own accessibility reviews

Generate **fresh classifications from review text** with keyword rules, TF-IDF + Logistic Regression, or a hosted OpenAI model using **your own API key**. This notebook never loads the app's saved predictions or its historical cohort. The LLM option calls the OpenAI API; it does not run a local LLM.

**Quick start:** choose **Run → Run All Cells** for an invented 120-review example. Then upload your own reviews CSV through JupyterLab's file browser, change `INPUT_CSV` below, and run all cells again. The example is for learning the workflow; its results are not research findings.

Binder sessions are temporary. Download the results ZIP before you leave. Use data suitable for public Binder. For LLM classification, enter your API key only in the masked prompt, never in a code cell. Public Binder is a shared cloud service; masked entry does not provide the same control as your own machine. [Binder limits and privacy](https://mybinder.readthedocs.io/en/latest/about/user-guidelines.html).


## 1. Choose your data and method

Only `place_id` and `review` columns are required in a UTF-8 CSV. Each location needs a stable `place_id`. Optional columns: `place_name` (or `name`), `address`, `review_date`, `service` (`clinic`, `pharmacy`, or `grocery`), and `category`. To reclassify the organizer data, download the original [Reviews CSV](https://drive.google.com/file/d/1Ov_U0PP9i7DYmrtvnbPBLtboDSOo1SwB/view), upload it here as `google_maps_reviews.csv`, and set `INPUT_CSV` accordingly. The original full-file SHA-256 is `1787435e1efbaddd88256efe38d481cec2b442e754bbe3d6b9e9e03d7a8674d0`; compare it with the checksum in the input audit. The original organizer Reviews CSV works directly; reviewer profile fields and owner replies are ignored. This is a reviews-only workflow: there is no Places join, geocoding, closure verification, or recreation of the old 79-location sample.

- `METHOD = "keywords"`: run the documented keyword baseline without training.
- `METHOD = "tfidf_logreg"`: fit fresh TF-IDF + Logistic Regression classifiers. At least five retained locations are required for a location-held-out test split.
- `METHOD = "llm"`: select `OPENAI_MODEL`, set a small `MAX_REVIEWS` (for example 20), and enable `ALLOW_PAID_API_CALLS`. The notebook prompts for your own key at runtime and sends one review per request to OpenAI. Your API account is billed. The model must support Responses API structured outputs.
- `TRAINING_LABEL_SOURCE = "keyword_rules"`: generate **provisional** training labels from rules. A fitted model is learning those rules; this does not establish real-world accuracy.
- `TRAINING_LABEL_SOURCE = "human"`: train using your own complete human annotations from `LABELS_CSV`.

The sample defaults to 2,000 unique reviews (maximum 10,000). Sampling is deterministic for the same file bytes/order, seed, and limit. To process more data, use the original full analysis notebook locally. Set `INPUT_CSV` to the uploaded file's relative path, for example `"google_maps_reviews.csv"`.


In [ ]:
INPUT_CSV = ""  # Empty = generate an invented example, not our saved results.
METHOD = "keywords"  # "keywords", "tfidf_logreg", or "llm"
TRAINING_LABEL_SOURCE = "keyword_rules"  # "keyword_rules" or "human"
LABELS_CSV = ""  # Optional completed human annotation CSV.
MAX_REVIEWS = 2000
SEED = 42
OUTPUT_DIR = "outputs/workshop"
OPENAI_MODEL = "gpt-5.6-terra"  # Choose a compatible model available to your API account.
ALLOW_PAID_API_CALLS = False  # Set True only when you want new OpenAI API requests.
LLM_MAX_NEW_REQUESTS = 20  # Hard request-count limit; not a dollar-cost estimate.


In [ ]:
from pathlib import Path
import json
import sys

ROOT = Path.cwd()
if not (ROOT / "access_app/workshop.py").is_file():
    raise RuntimeError("Open this notebook from the repository root so access_app/workshop.py is available.")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import pandas as pd
from IPython.display import FileLink, Markdown, display
from access_app.schema import ASPECTS
from access_app.workshop import (
    annotation_template, classify_reviews, export_run, keyword_rubric,
    load_human_labels, load_reviews, split_reviews, write_demo_csv, prompt_llm_classification,
)

if METHOD not in {"keywords", "tfidf_logreg", "llm"}:
    raise ValueError("METHOD must be keywords, tfidf_logreg, or llm.")
RUN_DIR = Path(OUTPUT_DIR) / METHOD
RUN_DIR.mkdir(parents=True, exist_ok=True)
synthetic_input = not bool(INPUT_CSV.strip())
input_path = write_demo_csv(RUN_DIR / "invented_input.csv") if synthetic_input else Path(INPUT_CSV)
print("Input:", "INVENTED EXAMPLE" if synthetic_input else input_path)
print("Method:", METHOD, "| Seed:", SEED)


## 2. Load, clean, sample, and split

The loader streams the CSV, ignores unrelated columns, removes empty records and global normalized exact-text duplicates, and samples eligible reviews using the fixed seed. Reviews longer than 65,536 characters are excluded and counted rather than silently truncated. Near duplicates are not removed.

The split holds out 20% of **locations**, so one location cannot appear in both training and test data. With fewer than five locations, keyword classification still works but there is no held-out evaluation. TF-IDF vocabulary and classifier fitting use training reviews only; hyperparameters are fixed before evaluation.


In [ ]:
reviews, audit = load_reviews(input_path, max_reviews=MAX_REVIEWS, seed=SEED)
reviews = split_reviews(reviews, seed=SEED)
display(pd.Series(audit, name="Input audit").to_frame())
display(reviews.groupby("split").agg(reviews=("record_id", "size"), locations=("place_id", "nunique")))
display(reviews[["record_id", "place_id", "review", "split"]].head(8))
reviews[["record_id", "place_id", "split"]].to_csv(RUN_DIR / "split_manifest.csv", index=False)


## 3. Inspect the rubric and optionally add human annotations

Each review receives one label for each aspect: Physical, Sensory, Language, Financial (Cost), and Time. States are `no_mention`, `barrier_reported`, `positive_access`, and `unclear_or_mixed`. **No mention does not mean accessible.** Keyword rules are deliberately simple and can miss context, negation, and access issues outside their vocabulary. The actual patterns are displayed below and exported with the results.

For human training or evaluation, download the blank annotation template, fill all five rows per review, and upload the completed file. Use an exact source quotation as `evidence` for mentioned aspects; leave it empty for `no_mention`. Fill `annotator_id`, `provenance=human`, and `adjudicated=true` only after human annotation/adjudication. The code checks structure and source quotations, but cannot verify that a human performed the annotation.

Set `LABELS_CSV` above and rerun with the **same input, seed, and sample limit**. Only complete five-aspect reviews are used. Training uses training-partition annotations; evaluation uses held-out test annotations. Reserve test annotations for independent evaluation and do not tune methods after inspecting test results. Labels copied from model predictions or keyword rules cannot establish independent accuracy.


In [ ]:
rubric, rubric_hash = keyword_rubric()
display(pd.Series(rubric["TERMS"], name="Aspect pattern").to_frame())
display(pd.Series({name: rubric[name] for name in ["NEG", "POS", "MIX"]}, name="State pattern").to_frame())
template_path = RUN_DIR / "annotation_template.csv"
annotation_template(reviews).to_csv(template_path, index=False)
display(FileLink(str(template_path)))
human_labels = load_human_labels(LABELS_CSV or None, reviews)
print("Complete supplied human-labeled reviews:", len(human_labels))


## 4. Classify from scratch

TF-IDF uses word unigrams/bigrams, up to 20,000 features, and class-balanced Logistic Regression (`C=1`, `max_iter=1000`). An aspect with only one observed training class uses an explicitly reported constant predictor. Missing training classes are shown below; they are not validated capabilities.

Evidence is copied from a contiguous span of rule-relevant sentences, or the full original review when a statistical prediction has no keyword match. The excerpt is for manual inspection: a source-faithful quotation does not prove the classification is correct. For `llm`, the model independently classifies your original reviews using the accessibility rubric and a strict output schema. It selects numbered source passages; the application copies and validates the exact quotations. It receives no keyword predictions or human labels. This is inference, not LLM fine-tuning.

The LLM option checks the number of uncached reviews against your request limit before asking for a key or making calls. Results are checkpointed after each validated response. Rerun to resume after a timeout or API error; failures never become `no_mention`. No automatic retries are made. Completed results can be replayed from your checkpoints without a key or new billing. Changing input text, model, rubric, or schema invalidates matching checkpoints.

The key is entered through `getpass`, held in function memory for the run, and omitted from notebook source, displayed outputs, files, and exports. It is not set as an environment variable. Restart the kernel after a paid run to release session memory. Your review text is sent to OpenAI with `store=false`; public Binder remains a shared hosted environment. [Structured outputs documentation](https://developers.openai.com/api/docs/guides/structured-outputs?api-mode=responses).


In [ ]:
if METHOD == "llm":
    predictions, classification = prompt_llm_classification(
        reviews, model=OPENAI_MODEL, cache_dir=RUN_DIR / "cache",
        max_requests=LLM_MAX_NEW_REQUESTS, allow_paid_calls=ALLOW_PAID_API_CALLS,
    )
else:
    predictions, classification = classify_reviews(
        reviews, method=METHOD, label_source=TRAINING_LABEL_SOURCE,
        human_labels=human_labels, seed=SEED,
    )
display(predictions.groupby(["aspect", "state"]).size().unstack(fill_value=0))
if classification["training_support"]:
    display(pd.DataFrame(classification["training_support"]).T)
display(predictions[predictions.state.ne("no_mention")][["record_id", "aspect", "state", "evidence"]].head(12))


## 5. Evaluate and download your results

Metrics are computed only when you supply complete human annotations for held-out locations. Without them, the report says **NOT EVALUATED**. Rule agreement and the invented example are not independent evidence of accuracy.

The ZIP includes your retained input text, all five classifications per review, source quotations, split manifest, blank annotation template, keyword rubric, evaluation report, and run metadata (source checksum, label-file checksum, seed, configuration, code hashes, Git revision when available, and package versions). For LLM runs, the ZIP also includes the classification rubric, response/model IDs, token usage, and checkpoints in `cache/`. To resume in a new session, extract the ZIP into `outputs/workshop/llm/`, restore the same source data and settings, and rerun. Retain the original input file and completed labels alongside this ZIP to rerun exactly the same sample. Fresh LLM calls can return different labels; checkpoints preserve your completed run. Normalized exact deduplication still allows near duplicates, and review sampling does not establish population prevalence or current accessibility.

For supported service categories, `frontend_results.json` can be imported into the local app. All categories remain in `classifications.csv`; unsupported categories are excluded only from the app export and counted in its coverage note. Coordinates are unknown, so imported locations appear in the list/evidence view without mapped points. To import, stop the local app, run `python -m access_app import /path/to/frontend_results.json`, then restart it.


In [ ]:
metadata, archive = export_run(
    reviews, predictions, audit, classification, human_labels, RUN_DIR,
    seed=SEED, synthetic=synthetic_input, labels_path=LABELS_CSV or None,
)
print(metadata["evaluation"]["status"])
if metadata["evaluation"]["aspects"]:
    display(pd.DataFrame({
        aspect: result["macro avg"]
        for aspect, result in metadata["evaluation"]["aspects"].items()
    }).T)
display(Markdown("**Download the ZIP now — Binder does not preserve your files after the session ends.**"))
display(FileLink(str(archive)))
print("Results directory:", RUN_DIR)


## Next steps

Try another method with unchanged source data, seed, and sample limit to compare predictions. Download each method's ZIP separately. Compare accuracy only using independent human annotations from reserved test locations. The app's precomputed results are never used as labels or predictions here.
